In [3]:
import cv2
import math
import numpy as np
import time
from realsense_camera import *

# Инициализация камеры
camera = RealsenseCamera()

# Переменные для трекинга
cx, cy = 0, 0
cx_mm, cy_mm = 0, 0
prev_cx, prev_cy = 0, 0
stable_count = 0

# Параметры стабилизации
MIN_STABLE_FRAMES = 5
MAX_MOVEMENT = 15

# Параметры фильтрации контуров
MIN_CONTOUR_AREA = 500
MIN_CONTOUR_WIDTH = 20
MIN_CONTOUR_HEIGHT = 20

# Параметры камеры (должны быть получены из калибровки)
Cx = 320  # Оптический центр X
Cy = 240  # Оптический центр Y
fx = 604.602  # Фокусное расстояние X
fy = 604.162  # Фокусное расстояние Y

print("Запуск обработки видео. Нажмите 'q' для выхода.")

while True:
    # Получаем кадры
    ret, depth_frame, color_frame, depth = camera.get_frame_stream()

    if not ret:
        print("Ошибка получения кадра")
        continue

    height, width = color_frame.shape[0:2]   

    # Обработка цвета
    hsv = cv2.cvtColor(color_frame, cv2.COLOR_BGR2HSV)    

    # Цветовые диапазоны
    bin1 = cv2.inRange(hsv, (20, 100, 100), (30, 255, 255)) # желтый
    bin2 = cv2.inRange(hsv, (0, 56, 95), (71, 255, 255)) # красный
    bin3 = cv2.inRange(hsv, (35, 100, 100), (85, 255, 255)) # зеленый
    bin4 = cv2.inRange(hsv, (90, 100, 100), (130, 255, 255)) # синий
    binary = bin1 + bin2 + bin3 + bin4  

    # Морфологические операции
    kernel = np.ones((5, 5), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Поиск контуров
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Фильтрация контуров
    valid_contours = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < MIN_CONTOUR_WIDTH or h < MIN_CONTOUR_HEIGHT:
            continue

        valid_contours.append(contour)

    # Обработка найденных контуров
    frame_display = color_frame.copy()

    if valid_contours:
        largest_contour = max(valid_contours, key=cv2.contourArea)
        moments = cv2.moments(largest_contour)

        if moments["m00"] > 0: 
            new_cx = int(moments["m10"] / moments["m00"])  
            new_cy = int(moments["m01"] / moments["m00"])

            # Проверка стабильности
            movement = math.sqrt((new_cx - prev_cx)**2 + (new_cy - prev_cy)**2)

            if movement < MAX_MOVEMENT:
                stable_count += 1
            else:
                stable_count = 0

            # Обновление координат при стабильности
            if stable_count >= MIN_STABLE_FRAMES:
                cx, cy = new_cx, new_cy

                # Преобразование в миллиметры с проверкой глубины
                if 0 <= cy < depth.shape[0] and 0 <= cx < depth.shape[1]:
                    depth_value = depth[cy, cx]
                    if depth_value > 0:
                        cx_mm = (cx - Cx) * depth_value / fx
                        cy_mm = (cy - Cy) * depth_value / fy
                    else:
                        cx_mm, cy_mm = 0, 0
                else:
                    cx_mm, cy_mm = 0, 0

            prev_cx, prev_cy = new_cx, new_cy

            # Визуализация
            cv2.drawContours(frame_display, [largest_contour], -1, (0, 255, 0), 3)
            cv2.circle(frame_display, (cx, cy), 8, (0, 0, 255), -1)
            cv2.circle(frame_display, (cx, cy), 12, (255, 255, 255), 2)

            # Отображение информации
            cv2.putText(frame_display, f'X: {cx} px', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'Y: {cy} px', (10, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'X: {cx_mm:.1f} mm', (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame_display, f'Y: {cy_mm:.1f} mm', (10, 120), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            print(f"Pixel: ({cx}, {cy}), MM: ({cx_mm:.1f}, {cy_mm:.1f})")

    cv2.imshow('Color Frame', frame_display)

    # Выход по 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()

Запуск обработки видео. Нажмите 'q' для выхода.
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (431, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), MM: (0.0, 0.0)
Pixel: (430, 145), M

Pixel: (185, 131), MM: (0.0, 0.0)
Pixel: (185, 129), MM: (0.0, 0.0)
Pixel: (185, 130), MM: (0.0, 0.0)
Pixel: (186, 130), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (185, 130), MM: (0.0, 0.0)
Pixel: (185, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (185, 130), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (187, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (185, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (187, 132), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 132), MM: (0.0, 0.0)
Pixel: (187, 132), MM: (0.0, 0.0)
Pixel: (186, 131), MM: (0.0, 0.0)
Pixel: (186, 132), MM: (0.0, 0.0)
Pixel: (186, 132), MM: (0.0, 0.0)
Pixel: (187, 1

KeyboardInterrupt: 

In [2]:
import cv2
import math
import numpy as np
import time
from realsense_camera import *

# Инициализация камеры
camera = RealsenseCamera()

# Переменные для трекинга
cx, cy = 0, 0
cx_mm, cy_mm = 0, 0
prev_cx, prev_cy = 0, 0
stable_count = 0

# Параметры стабилизации
MIN_STABLE_FRAMES = 5
MAX_MOVEMENT = 15

# Параметры фильтрации контуров
MIN_CONTOUR_AREA = 500
MIN_CONTOUR_WIDTH = 20
MIN_CONTOUR_HEIGHT = 20

# Параметры камеры (должны быть получены из калибровки)
Cx = 320  # Оптический центр X
Cy = 240  # Оптический центр Y
fx = 604.602  # Фокусное расстояние X
fy = 604.162  # Фокусное расстояние Y

rotation=0
def getOrientation(pts, img):
    rect = cv2.minAreaRect(pts)
    width = int(rect[1][0])
    height = int(rect[1][1])
    angle = int(rect[2])
    if width < height:
        angle = 90- angle
    else:
        angle = 90+(90-angle)
    return angle,width, height

print("Запуск обработки видео. Нажмите 'q' для выхода.")

while True:
    # Получаем кадры
    ret, depth_frame, color_frame, depth = camera.get_frame_stream()

    if not ret:
        print("Ошибка получения кадра")
        continue

    height, width = color_frame.shape[0:2]   

    # Обработка цвета
    hsv = cv2.cvtColor(color_frame, cv2.COLOR_BGR2HSV)    

    # Цветовые диапазоны
    bin1 = cv2.inRange(hsv, (20, 100, 100), (30, 255, 255)) # желтый
    bin2 = cv2.inRange(hsv, (0, 59, 0), (21, 255, 255)) # красный
    bin3 = cv2.inRange(hsv, (35, 100, 100), (85, 255, 255)) # зеленый
    bin4 = cv2.inRange(hsv, (90, 100, 100), (130, 255, 255)) # синий
    binary = bin1 + bin2 + bin3 + bin4  

    # Морфологические операции
    kernel = np.ones((5, 5), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Поиск контуров
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Фильтрация контуров
    valid_contours = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < MIN_CONTOUR_WIDTH or h < MIN_CONTOUR_HEIGHT:
            continue

        valid_contours.append(contour)

    # Обработка найденных контуров
    frame_display = color_frame.copy()

    if valid_contours:
        largest_contour = max(valid_contours, key=cv2.contourArea)
        moments = cv2.moments(largest_contour)

        if moments["m00"] > 0: 
            new_cx = int(moments["m10"] / moments["m00"])  
            new_cy = int(moments["m01"] / moments["m00"])

            # Проверка стабильности
            movement = math.sqrt((new_cx - prev_cx)**2 + (new_cy - prev_cy)**2)

            if movement < MAX_MOVEMENT:
                stable_count += 1
            else:
                stable_count = 0

            # Обновление координат при стабильности
            if stable_count >= MIN_STABLE_FRAMES:
                cx, cy = new_cx, new_cy

                # Преобразование в миллиметры с проверкой глубины
                if 0 <= cy < depth.shape[0] and 0 <= cx < depth.shape[1]:
                    depth_value = depth[cy, cx]
                    if depth_value > 0:
                        cx_mm = (cx - Cx) * depth_value / fx
                        cy_mm = (cy - Cy) * depth_value / fy
                    else:
                        cx_mm, cy_mm = 0, 0
                else:
                    cx_mm, cy_mm = 0, 0

            prev_cx, prev_cy = new_cx, new_cy
            
            rotation,width,height=getOrientation(largest_contour, frame_display)
            
            # Визуализация
            cv2.drawContours(frame_display, [largest_contour], -1, (0, 255, 0), 3)
            cv2.circle(frame_display, (cx, cy), 8, (0, 0, 255), -1)
            cv2.circle(frame_display, (cx, cy), 12, (255, 255, 255), 2)

            # Отображение информации
            cv2.putText(frame_display, f'X: {cx} px', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'Y: {cy} px', (10, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'X: {cx_mm:.1f} mm', (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame_display, f'Y: {cy_mm:.1f} mm', (10, 120), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame_display, f'rotation: {rotation:.1f}', (10, 150), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)  

            print(f"Pixel: ({cx}, {cy}), MM: ({cx_mm:.1f}, {cy_mm:.1f})")
    cv2.imshow('Color Frame', frame_display)

    # Выход по 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
            
camera.release()
cv2.destroyAllWindows()

Запуск обработки видео. Нажмите 'q' для выхода.
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (0, 0), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), MM: (0.0, 0.0)
Pixel: (432, 144), M

KeyboardInterrupt: 

In [ ]:
import cv2
import math
import numpy as np
import time
from realsense_camera import *
from kuka import Kuka
from openshowvar import *

#Connect robot
robot = openshowvar(ip = '192.168.17.2', port = 7000)
kuka = Kuka(robot)

kuka.set_base(8)
kuka.set_tool(10)

# Set speed (%)
kuka.set_speed(50)

kuka.read_cartesian()
x0=282.73
y0=122.48
z0=451.23
# Чтение текущих координат робота
kuka.read_cartesian()
# Построение траектории
first_point=[x0, y0, z0, 90.57, 0, 180]
trajectory=np.array([first_point])
kuka.lin_continuous(kuka,trajectory)

# Инициализация камеры
camera = RealsenseCamera()

# Переменные для трекинга
cx, cy = 0, 0
cx_mm, cy_mm = 0, 0
stable_count = 0

# Параметры стабилизации
MIN_STABLE_FRAMES = 5
MAX_MOVEMENT = 15

# Параметры фильтрации контуров
MIN_CONTOUR_AREA = 500
MIN_CONTOUR_WIDTH = 20
MIN_CONTOUR_HEIGHT = 20

# Параметры камеры (должны быть получены из калибровки)
Cx = 320  # Оптический центр X
Cy = 240  # Оптический центр Y
fx = 604.602  # Фокусное расстояние X
fy = 604.162  # Фокусное расстояние Y
rotation=0
CX_mm=0
CY_mm=0

def getOrientation(pts, img):
    rect = cv2.minAreaRect(pts)
    width = int(rect[1][0])
    height = int(rect[1][1])
    angle = int(rect[2])
    if width < height:
        angle = 90-angle
    else:
        angle = 90+(90-angle)
    return angle,width, height

print("Запуск обработки видео. Нажмите 'q' для выхода.")

while True:
    # Получаем кадры
    ret, depth_frame, color_frame, depth = camera.get_frame_stream()

    if not ret:
        print("Ошибка получения кадра")
        continue

    height, width = color_frame.shape[0:2]   

    # Обработка цвета
    hsv = cv2.cvtColor(color_frame, cv2.COLOR_BGR2HSV)    

    # Цветовые диапазоны
    bin1 = cv2.inRange(hsv, (20, 100, 100), (30, 255, 255)) # желтый
    bin2 = cv2.inRange(hsv, (0, 59, 0), (21, 255, 255)) # красный
    bin3 = cv2.inRange(hsv, (35, 100, 100), (85, 255, 255)) # зеленый
    bin4 = cv2.inRange(hsv, (90, 100, 100), (130, 255, 255)) # синий
    binary = bin1 + bin2 + bin3 + bin4  

    # Морфологические операции
    kernel = np.ones((5, 5), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Поиск контуров
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Фильтрация контуров
    valid_contours = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < MIN_CONTOUR_WIDTH or h < MIN_CONTOUR_HEIGHT:
            continue

        valid_contours.append(contour)

    # Обработка найденных контуров
    frame_display = color_frame.copy()

    if valid_contours:
        largest_contour = max(valid_contours, key=cv2.contourArea)
        moments = cv2.moments(largest_contour)

        if moments["m00"] > 0: 
            cx = int(moments["m10"] / moments["m00"])  
            cy = int(moments["m01"] / moments["m00"])

            # Преобразование в миллиметры с проверкой глубины
            if 0 <= cy < depth.shape[0] and 0 <= cx < depth.shape[1]:
                depth_value = int(depth[cy, cx])
                if depth_value > 0:
                    cx_mm = (cx - Cx) * depth_value / fx
                    cy_mm = (cy - Cy) * depth_value / fy
                    rotation,width,height=getOrientation(largest_contour, frame_display)
                    CX_mm=cx_mm-30
                    CY_mm=cy_mm-65
                    depth_value=depth[cy, cx]-452

                    print(CX_mm, CY_mm, depth_value)

                    # Движение Kuka
                    kuka.open_grip()
                    # Чтение текущих координат робота
                    kuka.read_cartesian()
                    # Построение траектории
                    first_point=[kuka.x_cartesian+CX_mm, kuka.y_cartesian-CY_mm, depth_value,kuka.A_cartesian, 0, kuka.C_cartesian]
                    second_point=[kuka.x_cartesian+CX_mm, kuka.y_cartesian-CY_mm, depth_value,rotation, 0, kuka.C_cartesian]
                    third_point=[kuka.x_cartesian+CX_mm, kuka.y_cartesian-CY_mm, depth_value-30,rotation, 0, kuka.C_cartesian]
                    trajectory=np.array([first_point, second_point, third_point])
                    kuka.lin_continuous(kuka,trajectory)
                    kuka.close_grip()
                    time.sleep(1)
                    # Чтение текущих координат робота
                    kuka.read_cartesian()
                    # Построение траектории
                    first_point=[x0, y0, z0, 90.57, 0, 180]
                    trajectory=np.array([first_point])
                    kuka.lin_continuous(kuka,trajectory)
                    kuka.read_cartesian()
                    # Построение траектории
                    first_point=[kuka.x_cartesian+CX_mm, kuka.y_cartesian-CY_mm, depth_value,kuka.A_cartesian, 0, kuka.C_cartesian]
                    second_point=[kuka.x_cartesian+CX_mm, kuka.y_cartesian-CY_mm, depth_value-30,rotation, 0, kuka.C_cartesian]
                    trajectory=np.array([first_point, second_point])
                    kuka.lin_continuous(kuka,trajectory)
                    kuka.open_grip()
                    time.sleep(1)
                    # Чтение текущих координат робота
                    kuka.read_cartesian()
                    # Построение траектории
                    first_point=[x0, y0, z0, 90.57, 0, 180]
                    trajectory=np.array([first_point])
                    kuka.lin_continuous(kuka,trajectory)
                    
                else:
                    cx_mm, cy_mm = 0, 0
            else:
                cx_mm, cy_mm = 0, 0

            prev_cx, prev_cy = cx, cy
            
            
            
            # Визуализация
            cv2.drawContours(frame_display, [largest_contour], -1, (0, 255, 0), 3)
            cv2.circle(frame_display, (cx, cy), 8, (0, 0, 255), -1)
            cv2.circle(frame_display, (cx, cy), 12, (255, 255, 255), 2)

            # Отображение информации
            cv2.putText(frame_display, f'X: {cx} px', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'Y: {cy} px', (10, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            cv2.putText(frame_display, f'X: {cx_mm:.1f} mm', (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame_display, f'Y: {cy_mm:.1f} mm', (10, 120), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(frame_display, f'rotation: {rotation:.1f}', (10, 150), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)  

            print(f"Pixel: ({cx}, {cy}), MM: ({cx_mm:.1f}, {cy_mm:.1f})")
    cv2.imshow('Color Frame', frame_display)

    # Выход по 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
            
camera.release()
cv2.destroyAllWindows()

Запуск обработки видео. Нажмите 'q' для выхода.
-26.36785852511239 -65.90869667407087 97
Pixel: (324, 239), MM: (3.6, -0.9)
-87.31042900949716 34.13731747445222 43
Pixel: (250, 361), MM: (-57.3, 99.1)
-53.34097472386793 -35.802533095428046 52
Pixel: (292, 275), MM: (-23.3, 29.2)
-55.05780662320006 -33.236996037486634 53
Pixel: (290, 278), MM: (-25.1, 31.8)
-54.07865008716478 -32.594784180401945 50
